In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_config
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler # used to normalize data for radar charts # conda install -c conda-forge scikit-learn   
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw # pip install fastdtw

# To handle curvature issue 
# robot stagnate or slowly moves curvature explode
from scipy.signal import medfilt
from scipy.interpolate import interp1d

**Handling velocity**

In [ ]:
def model_only_segments(df, teleop_col='lin_x_teleop'):
    """Return boolean mask and segment indices for contiguous model-controlled segments."""
    mask_model_only = df[teleop_col] == 0.0
    # label contiguous True regions
    # diff on mask: True->False transitions produce boundaries
    idx = np.where(mask_model_only)[0]
    if len(idx) == 0:
        print("No model segments found")
        return []  # no model segments
    # find breaks where indices are not consecutive
    breaks = np.where(np.diff(idx) != 1)[0] # if difference between consecutive indices is not 1, we have a jump in indices meaning we used teleoperation
    segments = []
    start = idx[0]
    for b in breaks:
        end = idx[b]
        segments.append((start, end))
        start = idx[b+1]
    segments.append((start, idx[-1]))
    return segments, breaks

## Process data

In [ ]:
reference_header = 'reference'

config = load_config()
# TODO Handle now that we cannot be inside docker using plotly anymore

df = pd.read_csv(f"../dataframes/all_data_20251029-030058.csv") #Index(['pose_x', 'pose_y', 'goal', 'robot', 'environment', 'env_type','augmentation'],
df.head()

# Compute path lenght

In [ ]:
# Take into account the whole path itself (with intervention)

def compute_path_length(df: pd.DataFrame):
    path_lengths = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        positions = group[['pose_x', 'pose_y']].to_numpy()
        length = 0.0
        for i in range(1, len(positions)):
            length += euclidean(positions[i-1], positions[i])
        path_lengths.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'path_length': length
        })
    return pd.DataFrame(path_lengths)


In [ ]:
path_lengths_df = compute_path_length(df)
path_lengths_df.head()

In [ ]:
df = df.merge(path_lengths_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

# Compute number of intervention

In [ ]:
# Failsafe as some bags don't have teleop data because there was no intervention at all
def intervention_count(df: pd.DataFrame, teleop_col: str = 'lin_x_teleop'):
    intervention_counts = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:

        if teleop_col not in group.columns:
            intervention_count = 0        
        else:
            _, breaks = model_only_segments(group, teleop_col=teleop_col)
            intervention_count = len(breaks) if len(breaks) > 0 else 0

        intervention_counts.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'intervention_count': intervention_count
        })
    return pd.DataFrame(intervention_counts)

In [ ]:
intervention_counts_df = intervention_count(df, teleop_col='lin_x_teleop')
intervention_counts_df.head()

In [ ]:
df = df.merge(intervention_counts_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

# Computing arrival rate based on CARE.  **(FOR NOW NOT A RATE)**

**Arrival rate** is defined as: *# goal reach w or w/o collisions* for our case its intervention

In [ ]:
def count_arrival(df: pd.DataFrame):
    arrivals = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        goal_reached = group['goal'].any()
        arrivals.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'arrival': int(goal_reached)
        })
    return pd.DataFrame(arrivals)

In [ ]:
count_arrival_df = count_arrival(df)
count_arrival_df.head()

In [ ]:
df = df.merge(count_arrival_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

# Compute safe only arrivals

In [ ]:
def count_arrival_safe(df: pd.DataFrame):
    arrivals = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        goal_reached = group['goal'].any()

        teleop_used = group['lin_x_teleop'].any() if 'lin_x_teleop' in group.columns else False
        if teleop_used:
            goal_reached = False

        arrivals.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'arrival_safe': int(goal_reached)
        })
    return pd.DataFrame(arrivals)

In [ ]:
count_arrival_safe_df = count_arrival_safe(df)
count_arrival_safe_df.head()

In [ ]:
df = df.merge(count_arrival_safe_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

# Count false goal positive 
**When model predict arrrived at goal but far from ground truth**

In [ ]:
def compute_dist_to_goal(df: pd.DataFrame, reference_header: str = 'reference'):
    assert reference_header in df["augmentation"], "Reference path not found in dataframe."
    dist_to_goals = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        # Get reference goal position
        df_reference = df[df['augmentation'] == reference_header]
        ref_group = df_reference[
            (df_reference['robot'] == robot) &
            (df_reference['environment'] == environment) &
            (df_reference['augmentation'] == augmentation) &
            (df_reference['env_type'] == env_type)
        ]
        if ref_group.empty:
            print(f"No reference data for {robot}, {environment}, {augmentation}, {env_type}")
            continue
        goal_pos = ref_group[['goal_x', 'goal_y']].iloc[0].to_numpy()

        # Compute final distance to goal
        final_pos = group[['pose_x', 'pose_y']].iloc[-1].to_numpy()
        dist_to_goal = euclidean(final_pos, goal_pos)

        dist_to_goals.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'dist_to_goal': dist_to_goal
        })
    return pd.DataFrame(dist_to_goals)

In [ ]:
def count_false_goal_positives(df: pd.DataFrame, goal_threshold: float = 0.5):
    false_positives = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:

        
        if 'distance_to_goal' not in group.columns:
            dist_goal_df = compute_dist_to_goal(df)
            df = df.merge(dist_goal_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')


        goal_reached = group['goal'].any()
        final_distance = group['distance_to_goal'].iloc[-1]

        false_positive = goal_reached and (final_distance > goal_threshold)

        false_positives.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'false_goal_positive': int(false_positive)
        })
    return pd.DataFrame(false_positives)

## Add refence dataframe

In [ ]:
folder_path = "/home/mae/Documents/GIT/Research/SafeGNM/metrics/dataframes"
reference_header = 'reference'

grouped = df.groupby(['robot', 'environment'])
for (robot, environment), group in grouped:
    ref_df = pd.read_csv(f"{folder_path}/{robot}/{environment}/{reference_header}/{robot}_{environment}_{reference_header}_odom.csv")
    print(ref_df.head())

## Compute dist to goal

In [ ]:
df["augmentation"].unique()

In [ ]:
distance_to_goal_df = compute_dist_to_goal(df)
distance_to_goal_df.head()
df = df.merge(distance_to_goal_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

## Compute false goal positive